# 🤖 Binary Classification Template — v2
### A complete, runnable scaffold for supervised binary classification projects

---

## How to use this notebook

1. **Read the markdown cells first** — they explain *what* each step does and *why* it's needed
2. **Search for `# ← UPDATE`** comments — these are the only places you need to change for your own dataset
3. **Run cells top-to-bottom** — later sections depend on variables defined earlier
4. The dummy dataset at the top will be automatically replaced when you load your own CSV

---

## The full workflow at a glance

```
1. Imports & Config
2. Load Data
3. Feature Engineering
4. Exploratory Data Analysis (EDA)
5. Train / Test Split
6. Preprocessing Pipeline
7. Helper Functions
8. Model Zoo + 5-Fold Cross-Validation
9. Hyperparameter Tuning
10. Final Test Evaluation
11. Feature Importance
12. Per-Model Confusion Matrices & ROC Curves
13. Combined ROC Curve (all models on one chart)
14. SHAP Model Interpretability
15. Conclusion & Checklist
```

> **This notebook follows the same structure as:** `07-01-machine-learning.ipynb` and `07-02-california_housing-v2_v9999.ipynb`

---
## Step 1 — Imports & Configuration

We import everything at the top so there are no surprises later.

**Core libraries used:**
- `numpy`, `pandas` — data handling and manipulation
- `matplotlib`, `seaborn` — visualisation
- `sklearn` — all machine learning tools (models, pipelines, metrics, etc.)
- `xgboost`, `lightgbm` — high-performance gradient boosting models

**Configuration set here:**
- Colour palette constants for confusion matrix plots (`CLR_TP`, `CLR_TN`, etc.)
- `RANDOM_STATE = 123` — used everywhere for reproducibility (same seed = same results every run)
- `warnings.filterwarnings('ignore')` — suppresses noisy sklearn / lightgbm messages

In [ ]:
import time
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ── Model selection utilities ──────────────────────────────────────────────────
from sklearn.model_selection import (
    train_test_split,   # split data into train / test
    StratifiedKFold,    # k-fold that preserves class ratios in every fold
    cross_validate,     # run cross-validation and collect multiple metrics
    GridSearchCV,       # exhaustive hyperparameter search
)

# ── Preprocessing ──────────────────────────────────────────────────────────────
from sklearn.preprocessing import (
    StandardScaler,         # normalise numeric features to mean=0, std=1
    OneHotEncoder,          # convert categorical text → binary columns
    LabelEncoder,           # convert target labels → integers (required by XGBoost)
    FunctionTransformer,    # wrap a custom Python function as a sklearn transformer
)
from sklearn.impute import SimpleImputer          # fill in missing values
from sklearn.feature_selection import VarianceThreshold  # drop zero-variance columns

# ── Pipeline utilities ─────────────────────────────────────────────────────────
from sklearn.pipeline import Pipeline             # chain preprocessing + model into one object
from sklearn.compose import ColumnTransformer     # apply different transforms to different columns

# ── Evaluation metrics ─────────────────────────────────────────────────────────
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,      # area under the ROC curve — best overall metric for binary classification
    confusion_matrix,   # 2x2 table of TP / TN / FP / FN counts
    roc_curve,          # (false positive rate, true positive rate) pairs at all thresholds
    classification_report,
)

# ── Classifiers ───────────────────────────────────────────────────────────────
from sklearn.linear_model import LogisticRegression     # linear model — great interpretable baseline
from sklearn.tree import DecisionTreeClassifier          # single decision tree — prone to overfit
from sklearn.ensemble import RandomForestClassifier      # many trees averaged — robust ensemble
from sklearn.neighbors import KNeighborsClassifier       # predicts based on nearest training examples
from xgboost import XGBClassifier                        # gradient boosting — often top performer
from lightgbm import LGBMClassifier                     # faster gradient boosting variant

# ── Global config ─────────────────────────────────────────────────────────────
warnings.filterwarnings("ignore")
RANDOM_STATE = 123  # change this to any integer; keep it fixed for reproducibility

# Colour palette used in confusion matrix plots
CLR_TP   = "#2ecc71"   # green  — True Positives
CLR_TN   = "#3498db"   # blue   — True Negatives
CLR_FP   = "#e74c3c"   # red    — False Positives (model said yes, truth was no)
CLR_FN   = "#f39c12"   # orange — False Negatives (model said no, truth was yes)
CLR_CURVE = "#8e44ad"  # purple — ROC curve line
CLR_DARK  = "#2c3e50"  # dark   — general accent

print("✅ Imports and configuration ready.")

---
## Step 2 — Load Data

Replace the dummy dataset block with your own CSV (or other source).

**After loading, always inspect:**
- `df.shape` — how many rows × columns?
- `df.dtypes` — are numeric columns typed as numbers, and text columns as `object`?
- `df.head()` — do the values look sensible?
- `df.describe()` — are ranges / means reasonable?
- `df['target'].value_counts(normalize=True)` — is the dataset class-imbalanced?

> 📌 **Unit of analysis:** What does one row represent? A person? A transaction? A time period?
> This matters for how you interpret features and predictions.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# ← UPDATE: Replace the block below with your real data source.
# Example:  df = pd.read_csv("your_dataset.csv")
# ─────────────────────────────────────────────────────────────────────────────

# Dummy dataset for demonstration — delete from here ...
np.random.seed(RANDOM_STATE)
df = pd.DataFrame({
    "feature1":       np.random.randn(1000),
    "feature2":       np.random.rand(1000) * 100,
    "feature3":       np.random.randint(0, 50, 1000),
    "category1":      np.random.choice(["A", "B", "C", "Rare"], 1000, p=[0.4, 0.4, 0.18, 0.02]),
    "category2":      np.random.choice(["X", "Y"], 1000),
    "target_column":  np.random.choice([0, 1], 1000, p=[0.85, 0.15]),  # imbalanced: 85/15
})
# ... to here, once you have your own data.

# ─────────────────────────────────────────────────────────────────────────────
# Initial inspection — always run these
# ─────────────────────────────────────────────────────────────────────────────
print("Dataset shape:", df.shape)
print("\nColumn types:")
print(df.dtypes)
print("\nFirst 5 rows:")
display(df.head())
print("\nSummary statistics:")
display(df.describe())

---
## Step 3 — Feature Engineering

Create new features from raw columns *before* splitting. Common patterns:

| Transformation | When to use | Example |
|----------------|-------------|-------|
| **Ratio features** | Raw totals that depend on group size | `rooms / households` |
| **Log transform** | Right-skewed numeric features | `log1p(income)` |
| **Interaction terms** | Two features that combine non-linearly | `age × income` |
| **Binning** | Continuous → ordinal categories | `pd.cut(age, bins)` |
| **Date decomposition** | Datetime columns | `month`, `day_of_week`, `days_since` |

**If your target is continuous, binarise it here:**
```python
df['target'] = np.where(df['price'] >= 500_000, 'expensive', 'affordable')
df = df.drop(columns=['price'])  # CRITICAL: drop the source column — leaving it in is data leakage!
```

> ⚠️ **Leakage rule:** Immediately drop any column that was directly used to *derive* the target. If the model can see the source, it's cheating.

In [ ]:
# ← UPDATE: Add your feature engineering here if needed.
# Example:
# df['ratio_feature'] = df['col_a'] / df['col_b']
# df['log_income']    = np.log1p(df['income'])
# df['target']        = np.where(df['value_col'] >= THRESHOLD, 'above', 'below')
# df = df.drop(columns=['value_col'])  # ← drop the source column!

# If using the dummy data, nothing to do here.
print("✅ Feature engineering complete. Shape:", df.shape)

---
## Step 4 — Exploratory Data Analysis (EDA)

**Never fit a model on data you haven't looked at.** EDA answers four questions:

1. **Class balance** — is the target skewed? (affects metric choice)
2. **Missing values** — do we need to impute or drop?
3. **Feature distributions** — any obvious predictors? Any extreme outliers?
4. **Correlations** — are any features nearly identical (multicollinearity)?

> 🎯 A model that always predicts the majority class can score high *accuracy* while learning nothing. Always use **ROC-AUC** or **F1** as your primary metric when classes are imbalanced.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# ← UPDATE: Replace 'target_column' with your actual target column name here
TARGET_COL = "target_column"
# ─────────────────────────────────────────────────────────────────────────────

# --- 4.1  Class distribution ---------------------------------------------------
# Shows whether classes are balanced (50/50) or imbalanced (e.g. 90/10).
# Imbalanced classes → use class_weight='balanced' in models, report AUC not accuracy.
print("=" * 50)
print("4.1  Target (class) distribution")
print("=" * 50)
print(df[TARGET_COL].value_counts())
print("\nAs proportions:")
print(df[TARGET_COL].value_counts(normalize=True).round(3))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df[TARGET_COL].value_counts().plot(kind='bar', ax=axes[0], color=[CLR_TN, CLR_TP],
                                    title="Class Distribution (counts)", rot=0)
df[TARGET_COL].value_counts(normalize=True).plot(kind='bar', ax=axes[1], color=[CLR_TN, CLR_TP],
                                                  title="Class Distribution (proportions)", rot=0)
plt.tight_layout(); plt.show()

# --- 4.2  Missing values -------------------------------------------------------
# Tells you which columns have gaps and how many.
# Our preprocessing pipeline handles these automatically via SimpleImputer.
print("\n" + "=" * 50)
print("4.2  Missing values (% per column)")
print("=" * 50)
missing = (df.isnull().mean() * 100).round(2)
missing = missing[missing > 0]
print(missing if len(missing) > 0 else "No missing values found 🎉")

# --- 4.3  Numeric feature distributions by class ------------------------------
# Plots histogram of each numeric feature split by class.
# A feature where the two class distributions separate well → strong predictor.
numeric_cols = df.select_dtypes(include=np.number).columns.drop(TARGET_COL).tolist()
cat_cols     = df.select_dtypes(include="object").columns.tolist()

print("\n" + "=" * 50)
print("4.3  Numeric feature distributions by class")
print("=" * 50)
if numeric_cols:
    fig, axes = plt.subplots(1, len(numeric_cols), figsize=(5 * len(numeric_cols), 4))
    if len(numeric_cols) == 1:
        axes = [axes]
    for ax, col in zip(axes, numeric_cols):
        for cls, grp in df.groupby(TARGET_COL):
            grp[col].hist(ax=ax, alpha=0.6, label=f"class {cls}", bins=25)
        ax.set_title(col); ax.legend()
    plt.tight_layout(); plt.show()

# --- 4.4  Correlation matrix --------------------------------------------------
# Spearman correlation is more robust than Pearson for skewed data.
# Pairs > ±0.8 are nearly redundant — consider dropping one.
print("\n" + "=" * 50)
print("4.4  Correlation matrix (Spearman, numeric features)")
print("=" * 50)
if len(numeric_cols) >= 2:
    corr = df[numeric_cols].corr(method="spearman")
    plt.figure(figsize=(8, 5))
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1,
                linewidths=0.5)
    plt.title("Spearman Correlation Matrix")
    plt.tight_layout(); plt.show()
else:
    print("Need at least 2 numeric features for a correlation matrix.")

---
## Step 5 — Train / Test Split

We split the data **before any preprocessing**. This is critical:

```
1. Split  →  2. Fit preprocessor on train only  →  3. Transform train
                                                  →  4. Transform test (using train statistics)
```

If you fit the scaler or imputer on the *full* dataset before splitting, your training set "sees" test statistics — this is **data leakage** and inflates performance estimates.

**Key parameters:**
- `test_size=0.20` — 80% train, 20% test (adjust for very small/large datasets)
- `stratify=y` — ensures both splits have the same class ratio. Always use this for imbalanced targets.
- `random_state=RANDOM_STATE` — same split every run

In [ ]:
# Separate features (X) from the target (y)
X = df.drop(columns=[TARGET_COL])  # everything except the target
y = df[TARGET_COL]                  # only the target column

# Re-detect column types on features (excluding the target)
numeric_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols     = X.select_dtypes(include="object").columns.tolist()

print(f"Numeric features  ({len(numeric_cols)}): {numeric_cols}")
print(f"Categorical features ({len(cat_cols)}): {cat_cols}")

# ── Split ─────────────────────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,             # 80% train, 20% test
    random_state=RANDOM_STATE,  # reproducible
    stratify=y,                 # preserve class ratio in both halves
)

# ── Encode target to integers (required by XGBoost / LightGBM) ────────────────
# LabelEncoder maps classes alphabetically: 0 = first class, 1 = second class, etc.
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)   # fit on train, then transform
y_test_enc  = le.transform(y_test)        # transform test using SAME mapping

print(f"\nClass mapping: {dict(zip(le.classes_, le.transform(le.classes_)))}")
print(f"\nTrain size: {X_train.shape[0]:,} rows  |  Test size: {X_test.shape[0]:,} rows")
print(f"Train class ratio: {pd.Series(y_train_enc).value_counts(normalize=True).round(3).to_dict()}")
print(f"Test  class ratio: {pd.Series(y_test_enc ).value_counts(normalize=True).round(3).to_dict()}")

---
## Step 6 — Preprocessing Pipeline

We build a `Pipeline` for each feature type, then combine them in a `ColumnTransformer`.

**Numeric pipeline:** `SimpleImputer` → `VarianceThreshold` → `StandardScaler`
- *Imputer*: fills missing values with the median (robust to outliers)
- *VarianceThreshold*: drops columns with zero variance (constant columns are useless)
- *StandardScaler*: normalises to mean=0, std=1 (essential for Logistic Regression and KNN)

**Categorical pipeline:** `SimpleImputer` → rare-category collapser → `OneHotEncoder`
- *Imputer*: fills missing values with the most frequent value
- *Rare collapser*: merges tiny categories (< 3% frequency) into `'Other'` to avoid sparse dummies
- *OneHotEncoder*: converts text categories into binary (0/1) columns
  - `handle_unknown='ignore'`: gracefully handles categories seen in test but not training
  - `drop='first'`: removes one dummy per variable to avoid perfect multicollinearity

**Why Pipeline instead of manual transforms?**
- ✅ No leakage — scaler is fitted only on the current training fold during cross-validation
- ✅ Single `.fit()` / `.predict()` call
- ✅ Deployment-ready — the whole object can be pickled

In [ ]:
# ── Custom helper: collapse rare categorical values ────────────────────────────
# Categories that appear in < 3% of rows are merged into 'Other'.
# Without this, OneHotEncoder creates many near-empty columns that add noise.
def collapse_rare_categories(X_df, threshold=0.03):
    """Replace categories with frequency < threshold with 'Other'."""
    out = X_df.copy()
    for col in out.columns:
        counts = out[col].value_counts(normalize=True)
        rare   = counts[counts < threshold].index
        out[col] = out[col].apply(lambda x: "Other" if x in rare else x)
    return out

# Wrap the function so sklearn can use it inside a Pipeline
rare_collapser = FunctionTransformer(
    lambda X: collapse_rare_categories(pd.DataFrame(X, columns=cat_cols)),
    validate=False,
)

# ── Numeric sub-pipeline ──────────────────────────────────────────────────────
num_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),     # fill NaNs with median
    ("zv",     VarianceThreshold(threshold=0)),       # drop zero-variance columns
    ("scale",  StandardScaler()),                     # normalise
])

# ── Categorical sub-pipeline ──────────────────────────────────────────────────
cat_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),     # fill NaNs with mode
    ("rare",   rare_collapser),                             # merge tiny categories
    ("ohe",    OneHotEncoder(
                   handle_unknown="ignore",   # unknown test categories → all zeros
                   sparse_output=False,       # return dense array
                   drop="first",              # avoid dummy variable trap
              )),
])

# ── Combine into one ColumnTransformer ────────────────────────────────────────
# Applies num_pipe to numeric columns and cat_pipe to categorical columns in parallel.
preprocessor = ColumnTransformer([
    ("num", num_pipe, numeric_cols),
    ("cat", cat_pipe, cat_cols),
])

print("✅ Preprocessing pipeline ready.")

---
## Step 7 — Helper Functions

We define reusable plotting functions **once** here, then call them for every model.
Consistent plots make it easy to compare models visually.

**Functions defined:**
- `get_outcomes` — labels each prediction as TP / TN / FP / FN
- `plot_confusion_matrix` — colour-coded 2×2 confusion matrix with metrics in the title
- `plot_roc_curve` — single ROC curve on given axes (reused in the combined plot)
- `plot_feature_importance` — horizontal bar chart of feature importances (works for trees and linear models)

In [ ]:
def get_outcomes(y_true, y_pred):
    """
    Label each prediction as 'TP', 'TN', 'FP', or 'FN'.
    Useful for scatter plots that colour-code prediction outcomes.
    """
    return np.where(
        y_pred == y_true,
        np.where(y_pred == 1, "TP", "TN"),
        np.where(y_pred == 1, "FP", "FN"),
    )


def plot_confusion_matrix(ax, y_true, y_pred, title, y_proba=None):
    """
    Draw a colour-coded 2×2 confusion matrix on the given axes.

    Colours: green=TP, red=FP, orange=FN, blue=TN
    The x-axis label shows Accuracy, Precision, Recall, and (optionally) AUC.
    """
    cm = confusion_matrix(y_true, y_pred)
    TN, FP, FN, TP = cm.ravel()    # unpack the four cells

    # Layout: top-left=TP, top-right=FN, bottom-left=FP, bottom-right=TN
    cell_colours = [[CLR_TP, CLR_FN], [CLR_FP, CLR_TN]]
    cell_labels  = [[f"TP\n{TP}", f"FN\n{FN}"], [f"FP\n{FP}", f"TN\n{TN}"]]

    ax.set_xlim(0, 2); ax.set_ylim(0, 2)
    for r in range(2):
        for c in range(2):
            ax.add_patch(plt.Rectangle((c, 1 - r), 1, 1,
                                       color=cell_colours[r][c], alpha=0.78))
            ax.text(c + 0.5, 1 - r + 0.5, cell_labels[r][c],
                    ha="center", va="center", fontsize=15,
                    fontweight="bold", color="white")

    ax.set_xticks([0.5, 1.5])
    ax.set_xticklabels(["Predicted: (1)", "Predicted: (0)"], fontsize=9)
    ax.set_yticks([0.5, 1.5])
    ax.set_yticklabels(["Actual: (0)", "Actual: (1)"], fontsize=9)
    ax.set_title(title, fontsize=12, fontweight="bold", pad=8)

    precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    recall    = TP / (TP + FN) if (TP + FN) > 0 else 0
    accuracy  = (TP + TN) / len(y_true)

    xlabel = f"Accuracy={accuracy:.2f}  Precision={precision:.2f}  Recall={recall:.2f}"
    if y_proba is not None:
        xlabel += f"  AUC={roc_auc_score(y_true, y_proba):.3f}"
    ax.set_xlabel(xlabel, fontsize=9, labelpad=8)


def plot_roc_curve(ax, y_true, y_proba, model_name, color=CLR_CURVE):
    """
    Plot a ROC curve on the given axes.

    The ROC curve shows the trade-off between sensitivity (TPR) and
    1-specificity (FPR) at every possible decision threshold.
    A perfect model has AUC=1.0; random guessing gives AUC=0.5.
    """
    fpr, tpr, _ = roc_curve(y_true, y_proba)
    auc = roc_auc_score(y_true, y_proba)
    ax.plot(fpr, tpr, color=color, lw=2.5,
            label=f"{model_name} (AUC={auc:.3f})")
    ax.fill_between(fpr, tpr, alpha=0.06, color=color)  # light shaded area under curve
    return auc


def plot_feature_importance(pipeline, top_n=15):
    """
    Extract and plot the top N feature importances from a fitted sklearn Pipeline.

    Works for:
    - Tree models (RandomForest, XGBoost, LightGBM) — uses .feature_importances_
    - Linear models (Logistic Regression)            — uses absolute .coef_ values
    """
    pre = pipeline.named_steps["pre"]
    clf = pipeline.named_steps["clf"]

    # Get feature names after all transformations (includes OHE-expanded columns)
    try:
        feature_names = pre.get_feature_names_out()
    except AttributeError:
        feature_names = [f"Feature_{i}" for i in range(X_train.shape[1])]

    # Get importance values based on model type
    if hasattr(clf, "feature_importances_"):
        importances  = clf.feature_importances_
        metric_label = "Feature Importance"
    elif hasattr(clf, "coef_"):
        importances  = np.abs(clf.coef_[0])  # absolute value — direction doesn't matter here
        metric_label = "Absolute Coefficient"
    else:
        print(f"⚠️  Feature importance not supported for {clf.__class__.__name__}")
        return

    # Build a tidy DataFrame; strip pipeline prefixes from names (e.g. 'num__feature1' → 'feature1')
    imp_df = pd.DataFrame({"Feature": feature_names, "Importance": importances})
    imp_df["Feature"] = imp_df["Feature"].str.split("__").str[-1]
    imp_df = imp_df.sort_values("Importance", ascending=True).tail(top_n)

    plt.figure(figsize=(10, 6))
    plt.barh(imp_df["Feature"], imp_df["Importance"], color=CLR_TN)
    plt.title(f"Top {top_n} {metric_label}s — {clf.__class__.__name__}",
              fontweight="bold")
    plt.xlabel(metric_label)
    plt.tight_layout()
    plt.show()


print("✅ Helper functions ready: get_outcomes, plot_confusion_matrix, plot_roc_curve, plot_feature_importance")

---
## Step 8 — Model Zoo + 5-Fold Cross-Validation

### 8.1 Why start with multiple models?

Different algorithms have different strengths. We run all of them first with default settings
so we know which family is worth tuning.

| Algorithm | Family | Needs scaling? | Notes |
|-----------|--------|---------------|-------|
| Logistic Regression | Linear | ✅ Yes | Interpretable, fast, strong baseline |
| Decision Tree | Tree | ❌ No | Easy to visualise but overfits |
| **Random Forest** | Ensemble (Bagging) | ❌ No | Robust, rarely overfits badly |
| **XGBoost** | Ensemble (Boosting) | ❌ No | Often the top performer |
| **LightGBM** | Ensemble (Boosting) | ❌ No | Faster than XGBoost on large data |
| KNN | Instance-based | ✅ Yes | Slow on large data, good reference |

### 8.2 Why 5-fold cross-validation instead of a single train/val split?

A single split gives **one noisy estimate**. With 5-fold CV:
- Every observation is in the *test* set exactly once
- We average 5 estimates → much more stable performance number
- We can also look at the standard deviation to detect unstable models

**Primary metric:** `roc_auc` — threshold-agnostic, insensitive to class imbalance  
**Secondary metric:** `f1_weighted` — threshold-dependent (at 0.5), accounts for class frequency

In [ ]:
# ── Define the model zoo ──────────────────────────────────────────────────────
# All models use the same RANDOM_STATE for reproducibility.
# class_weight is not set here — we'll add 'balanced' during tuning if needed.
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Decision Tree":        DecisionTreeClassifier(random_state=RANDOM_STATE),
    "Random Forest":        RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1),
    "KNN":                  KNeighborsClassifier(n_neighbors=5),
    "XGBoost":              XGBClassifier(eval_metric="logloss", random_state=RANDOM_STATE),
    "LightGBM":             LGBMClassifier(random_state=RANDOM_STATE, verbose=-1),
}
print(f"✅ {len(models)} baseline models defined.")

# ── 5-fold stratified cross-validation ───────────────────────────────────────
# StratifiedKFold ensures each fold has the same class ratio as the full dataset.
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_results = []

print("\nRunning 5-fold CV for each model (this may take ~30 seconds)...")
for name, clf in models.items():
    # Wrap preprocessor + model into a single Pipeline.
    # This ensures the preprocessor is refit on each training fold, preventing leakage.
    pipe = Pipeline([("pre", preprocessor), ("clf", clf)])

    cv_scores = cross_validate(
        pipe,
        X_train,
        y_train_enc,
        cv=cv5,
        scoring=["roc_auc", "f1_weighted"],
        n_jobs=-1,   # use all CPU cores
    )

    cv_results.append({
        "Model":    name,
        "Mean AUC": cv_scores["test_roc_auc"].mean(),
        "Std AUC":  cv_scores["test_roc_auc"].std(),
        "Mean F1":  cv_scores["test_f1_weighted"].mean(),
    })

results_df = pd.DataFrame(cv_results).sort_values("Mean AUC", ascending=False)

print("\n" + "=" * 55)
print("CV RESULTS (BASELINE, 5-FOLD, SORTED BY AUC)")
print("=" * 55)
print(results_df.round(4).to_string(index=False))

# Visualise as a bar chart with error bars (std deviation across folds)
fig, ax = plt.subplots(figsize=(9, 5))
colors = [CLR_TP if i == 0 else CLR_TN for i in range(len(results_df))]
ax.barh(results_df["Model"], results_df["Mean AUC"],
        xerr=results_df["Std AUC"], align="center", color=colors,
        alpha=0.85, capsize=5)
ax.set_xlabel("Mean ROC-AUC (5-fold CV)", fontsize=11)
ax.set_title("Baseline Model Comparison — 5-Fold CV", fontweight="bold")
ax.axvline(0.5, linestyle="--", color="gray", alpha=0.6, label="Random chance (AUC=0.5)")
ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

---
## Step 9 — Hyperparameter Tuning

We pick the **top 3 models** from CV and search for better hyperparameters using `GridSearchCV`.

**Key hyperparameters explained:**

| Parameter | Model | What it controls |
|-----------|-------|------------------|
| `C` | Logistic Regression | Regularisation strength — small C → more regularised |
| `n_estimators` | RF / XGBoost | Number of trees — more = slower but smoother |
| `max_depth` | RF / XGBoost | Maximum tree depth — lower = less overfit |
| `learning_rate` | XGBoost | Step size per tree — smaller = more trees needed |
| `min_samples_split` | Random Forest | Min samples to split a node — higher = more regularised |

> ⚠️ Always tune with `scoring='roc_auc'`, not `'accuracy'`. Using accuracy as the tuning objective optimises for the majority class and gives misleading results on imbalanced data.

In [ ]:
# ← UPDATE: Add or remove models from tuning_configs based on your CV results.
# Tip: tune the top 2–3 models from the CV leaderboard above.
tuning_configs = {
    "LogisticRegression": {
        "model": LogisticRegression(random_state=RANDOM_STATE, class_weight="balanced"),
        "params": {
            "clf__C":       [0.1, 1, 10],   # regularisation — lower = stronger penalty
            "clf__penalty": ["l2"],          # L2 is the standard; try 'l1' for sparse coefficients
            "clf__solver":  ["lbfgs"],       # default solver, works well for most sizes
        },
    },
    "RandomForest": {
        "model": RandomForestClassifier(random_state=RANDOM_STATE, class_weight="balanced"),
        "params": {
            "clf__n_estimators":     [100, 200],     # more trees = more stable but slower
            "clf__max_depth":        [5, 10, None],  # None = no limit (may overfit)
            "clf__min_samples_split": [2, 5],        # higher = more regularised
        },
    },
    "XGBoost": {
        "model": XGBClassifier(random_state=RANDOM_STATE, eval_metric="logloss"),
        "params": {
            "clf__n_estimators":  [100, 200],
            "clf__learning_rate": [0.05, 0.1],    # smaller = more trees needed but often better
            "clf__max_depth":     [3, 5],          # shallower trees reduce overfitting
        },
    },
}

tuning_results = []
grid_objects   = {}  # store fitted GridSearchCV objects for later

print("Starting Grid Search across 3 models...")
print("=" * 55)
for name, config in tuning_configs.items():
    tuning_pipeline = Pipeline([("pre", preprocessor), ("clf", config["model"])])

    grid_search = GridSearchCV(
        tuning_pipeline,
        config["params"],
        cv=cv5,
        scoring="roc_auc",   # ← always use AUC for binary classification tuning
        n_jobs=-1,
        verbose=0,
    )

    start_t = time.time()
    grid_search.fit(X_train, y_train_enc)
    elapsed = time.time() - start_t

    grid_objects[name] = grid_search
    tuning_results.append({
        "Model":    name,
        "Best AUC": grid_search.best_score_,
        "Time (s)": round(elapsed, 1),
        "Best Params": grid_search.best_params_,
    })
    print(f"[✓] {name:<22} {elapsed:>5.1f}s  |  Best CV AUC: {grid_search.best_score_:.4f}")
    print(f"    Best params: {grid_search.best_params_}")

# Identify the overall best model
best_model_name = sorted(tuning_results, key=lambda x: x["Best AUC"], reverse=True)[0]["Model"]
best_pipeline   = grid_objects[best_model_name].best_estimator_  # already fitted

print(f"\n🏆 Best model overall: {best_model_name}")

---
## Step 10 — Final Evaluation on the Test Set

**This is the moment of truth.**  
Everything so far (CV, tuning) was done on training data. The test set has **never been seen by any model**.

We evaluate the best tuned model on the held-out test set and report six metrics:

| Metric | When it matters most |
|--------|----------------------|
| **Accuracy** | Balanced classes only |
| **Precision** | Cost of false alarms is high (e.g. spam detection) |
| **Recall** | Cost of missing positives is high (e.g. cancer screening) |
| **F1-Score** | Need to balance precision and recall |
| **ROC-AUC** | Always — threshold-agnostic ranking metric |
| **Classification report** | Full breakdown per class |

> 💡 A gap between CV score and test score is normal. A *large* gap (> 5 AUC points) suggests the model overfit — consider more regularisation.

In [ ]:
print("=" * 55)
print(f"TEST SET RESULTS — {best_model_name}")
print("=" * 55)

# Generate predictions on the untouched test set
y_pred  = best_pipeline.predict(X_test)          # hard predictions (0 or 1)
y_proba = best_pipeline.predict_proba(X_test)[:, 1]  # probability of class 1

# Compute and print metrics
print(f"Accuracy:  {accuracy_score(y_test_enc, y_pred):.4f}")
print(f"Precision: {precision_score(y_test_enc, y_pred, zero_division=0):.4f}")
print(f"Recall:    {recall_score(y_test_enc, y_pred, zero_division=0):.4f}")
print(f"F1-Score:  {f1_score(y_test_enc, y_pred, zero_division=0):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test_enc, y_proba):.4f}")
print("\nFull Classification Report:")
print(classification_report(y_test_enc, y_pred, zero_division=0))

# Confusion matrix for the best model
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

plot_confusion_matrix(
    ax=axes[0],
    y_true=y_test_enc,
    y_pred=y_pred,
    title=f"{best_model_name} — Confusion Matrix (Test Set)",
    y_proba=y_proba,
)

plot_roc_curve(axes[1], y_test_enc, y_proba, best_model_name)
axes[1].plot([0, 1], [0, 1], "k--", alpha=0.5, label="Random Chance")
axes[1].set_title(f"{best_model_name} — ROC Curve (Test Set)", fontweight="bold")
axes[1].set_xlabel("False Positive Rate"); axes[1].set_ylabel("True Positive Rate")
axes[1].legend(loc="lower right")

plt.tight_layout(); plt.show()

---
## Step 11 — Feature Importance

Feature importance tells us **which inputs the model relied on most**.

- **Tree models** (Random Forest, XGBoost): `.feature_importances_` — how much each feature reduced impurity
- **Linear models** (Logistic Regression): `abs(coef_)` — how much each feature shifts the log-odds

> ⚠️ **Caveat:** Importance shows *which* features matter, not *in what direction*. Use SHAP (Step 14) for directional interpretation.

In [ ]:
# Plot top 10 most important features for the best tuned model
plot_feature_importance(best_pipeline, top_n=10)

---
## Step 12 — Confusion Matrices & ROC Curves for All Models

We refit all baseline models on the full training set and evaluate each on the test set.
This gives a side-by-side visual comparison of every model's error profile.

In [ ]:
print("Fitting all baseline models on training data and plotting test-set results...")

num_models = len(models)
fig, axes  = plt.subplots(nrows=num_models, ncols=2, figsize=(12, 5 * num_models))

# Ensure axes is always 2D (handles the edge case of 1 model)
if num_models == 1:
    axes = [axes]

for i, (name, clf) in enumerate(models.items()):
    # Build a fresh pipeline for this model and fit it
    pipe = Pipeline([("pre", preprocessor), ("clf", clf)])
    pipe.fit(X_train, y_train_enc)

    # Generate test-set predictions
    y_pred_i  = pipe.predict(X_test)
    y_proba_i = pipe.predict_proba(X_test)[:, 1]

    # ── Left column: Confusion Matrix ─────────────────────────────────────────
    plot_confusion_matrix(
        ax=axes[i][0],
        y_true=y_test_enc,
        y_pred=y_pred_i,
        title=f"{name} — Confusion Matrix",
        y_proba=y_proba_i,
    )

    # ── Right column: ROC Curve ───────────────────────────────────────────────
    plot_roc_curve(axes[i][1], y_true=y_test_enc, y_proba=y_proba_i, model_name=name)
    axes[i][1].plot([0, 1], [0, 1], "k--", alpha=0.5, label="Random Chance")
    axes[i][1].set_title(f"{name} — ROC Curve", fontsize=12, fontweight="bold", pad=8)
    axes[i][1].set_xlabel("False Positive Rate (1 - Specificity)", fontsize=9)
    axes[i][1].set_ylabel("True Positive Rate (Sensitivity)", fontsize=9)
    axes[i][1].legend(loc="lower right", fontsize=9)

plt.tight_layout(pad=3.0)
plt.show()

---
## Step 13 — Combined ROC Curve (All Models)

Overlaying all models on a single ROC plot makes it easy to see which model dominates
at every threshold — not just at 0.5. The closer a curve hugs the top-left corner, the better.

In [ ]:
plt.figure(figsize=(10, 8))
ax = plt.gca()

# Assign a distinct colour to each model
colors = plt.cm.tab10(np.linspace(0, 1, len(models)))

for (name, clf), color in zip(models.items(), colors):
    # Refit fresh pipeline — reusing pipes from Step 12 would also work,
    # but we rebuild here to keep this cell self-contained.
    pipe = Pipeline([("pre", preprocessor), ("clf", clf)])
    pipe.fit(X_train, y_train_enc)

    y_proba_i = pipe.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test_enc, y_proba_i)
    auc = roc_auc_score(y_test_enc, y_proba_i)

    ax.plot(fpr, tpr, lw=2.5, color=color, label=f"{name} (AUC = {auc:.3f})")

# Diagonal line = random guessing (AUC = 0.5)
ax.plot([0, 1], [0, 1], "k--", lw=2, label="Random Chance (AUC = 0.500)")

ax.set_title("Combined ROC Curve — Model Comparison", fontsize=14, fontweight="bold", pad=15)
ax.set_xlabel("False Positive Rate (1 - Specificity)", fontsize=11, labelpad=10)
ax.set_ylabel("True Positive Rate (Sensitivity)", fontsize=11, labelpad=10)
ax.set_xlim([-0.01, 1.0]); ax.set_ylim([0.0, 1.01])

# Sort legend entries by AUC (highest first)
handles, labels = ax.get_legend_handles_labels()
hl = sorted(zip(handles, labels), key=lambda x: x[1].split("=")[-1], reverse=True)
handles_sorted, labels_sorted = zip(*hl)
ax.legend(handles_sorted, labels_sorted, loc="lower right", fontsize=10, frameon=True, shadow=True)

ax.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

---
## Step 14 — SHAP: Model Interpretability

SHAP (SHapley Additive exPlanations) explains **why the model made each prediction**.

Unlike feature importance (which is a global average), SHAP assigns a contribution score
to each feature **for every individual prediction**.

**Three plots to always produce:**

| Plot | Code | Question answered |
|------|------|-------------------|
| **Beeswarm** | `shap.plots.beeswarm(...)` | Which features matter globally, and in which direction? |
| **Waterfall** | `shap.plots.waterfall(...)` | How was this one prediction built step by step? |
| **Dependence** | `shap.dependence_plot(...)` | How does one feature's effect vary across its range? |

**Reading SHAP values:**
- **Positive SHAP** → feature pushed prediction *toward class 1*
- **Negative SHAP** → feature pushed prediction *toward class 0*
- `E[f(X)]` = baseline (average model output across training set)
- `f(x)` = prediction for this sample = baseline + sum of all SHAP values

> 💡 `pip install shap` if not already installed.

In [ ]:
# ← UPDATE: uncomment and run these cells once 'shap' is installed.
# pip install shap

# import shap
#
# # Transform test/train sets using the fitted preprocessor step
# X_test_transformed  = best_pipeline.named_steps["pre"].transform(X_test)
# X_train_transformed = best_pipeline.named_steps["pre"].transform(X_train)
#
# # Recover clean feature names (strip pipeline prefixes like 'num__')
# try:
#     raw_names   = best_pipeline.named_steps["pre"].get_feature_names_out()
#     clean_names = [n.split("__")[-1] for n in raw_names]
# except AttributeError:
#     clean_names = [f"Feature_{i}" for i in range(X_train_transformed.shape[1])]
#
# X_test_display  = pd.DataFrame(X_test_transformed,  columns=clean_names)
# X_train_display = pd.DataFrame(X_train_transformed, columns=clean_names)
#
# # Use a sample of training data as the background for the explainer (speeds up computation)
# background = shap.sample(X_train_display, 100, random_state=RANDOM_STATE)
#
# # Create the SHAP explainer around the classifier step (not the full pipeline)
# explainer   = shap.Explainer(best_pipeline.named_steps["clf"], background)
# shap_values = explainer(X_test_display)
#
# # For binary classification, SHAP returns shape (n, features, 2) — slice class 1
# values_to_plot = shap_values[:, :, 1] if len(shap_values.shape) == 3 else shap_values
#
# print("✅ SHAP explainer ready.")

print("ℹ️  SHAP cells are commented out. Uncomment after installing: pip install shap")

In [ ]:
# SHAP Plot 1 — Beeswarm (global feature importance + direction)
# Each dot is one prediction. Position = SHAP value. Colour = feature value.
# Features are sorted by mean |SHAP| (most important at top).

# shap.plots.beeswarm(values_to_plot, max_display=12)

In [ ]:
# SHAP Plot 2 — Waterfall (single prediction explained step by step)
# Shows how each feature pushes the prediction up or down from the baseline.
# Change sample_idx to inspect a different prediction.

# sample_idx = 0
# shap.plots.waterfall(shap_values[sample_idx], max_display=12)

In [ ]:
# SHAP Plot 3 — Dependence (how one feature's SHAP value varies with its actual value)
# Reveals non-linear thresholds and interaction effects.
# 'interaction_index="auto"' colours by the feature that interacts most with the chosen one.

# top_feature = clean_names[0]  # ← UPDATE to the feature you want to inspect
# shap.dependence_plot(top_feature, values_to_plot.values, X_test_display,
#                      interaction_index="auto")

---
## Step 15 — Conclusion & Checklist

Replace ⬜ with ✅ as you complete each step.

| Step | Status |
|------|--------|
| Loaded and explored the dataset | ⬜ |
| Engineered new features (if applicable) | ⬜ |
| Ran EDA: class balance, missing values, distributions, correlations | ⬜ |
| Split data — stratified 80/20, split BEFORE preprocessing | ⬜ |
| Built preprocessing pipeline (impute → encode → scale) | ⬜ |
| Compared 6 classifiers using 5-fold CV | ⬜ |
| Tuned hyperparameters with GridSearchCV | ⬜ |
| Evaluated best model on held-out test set | ⬜ |
| Plotted confusion matrices and ROC curves | ⬜ |
| Analysed feature importance | ⬜ |
| Explained predictions using SHAP | ⬜ |

---

### Common pitfalls — double-check before submitting

| Pitfall | How to check |
|---------|--------------|
| **Data leakage** | Did you fit any preprocessor before splitting? |
| **Wrong label encoding** | Are XGBoost/LightGBM receiving integer labels? |
| **Test set contamination** | Did you ever use `y_test` to make modelling decisions? |
| **Reporting CV score as final score** | Did you evaluate the final model on held-out test data? |
| **Wrong metric for imbalanced data** | Using AUC/F1 rather than raw accuracy? |

---

### What to explore next

- **Class imbalance:** SMOTE oversampling, `class_weight='balanced'`
- **More powerful search:** `RandomizedSearchCV` for large parameter spaces
- **Feature selection:** `RFECV` (recursive elimination with cross-validation)
- **Ensemble stacking:** Use base model predictions as features for a meta-model
- **Threshold optimisation:** Adjust the 0.5 decision threshold to trade precision for recall
- **Deployment:** `joblib.dump(best_pipeline, 'model.pkl')` → load and serve via API
- **Calibration:** `CalibratedClassifierCV` if predicted probabilities need to be well-calibrated

In [ ]:
# Final performance summary — print a clean table of all key metrics
final_metrics = {
    "Best Model":  best_model_name,
    "Accuracy":    round(accuracy_score(y_test_enc, y_pred), 4),
    "Precision":   round(precision_score(y_test_enc, y_pred, zero_division=0), 4),
    "Recall":      round(recall_score(y_test_enc, y_pred, zero_division=0), 4),
    "F1-Score":    round(f1_score(y_test_enc, y_pred, zero_division=0), 4),
    "ROC-AUC":     round(roc_auc_score(y_test_enc, y_proba), 4),
}

summary_df = pd.DataFrame(list(final_metrics.items()), columns=["Metric", "Value"])
print("\n" + "=" * 40)
print("  FINAL PERFORMANCE SUMMARY")
print("=" * 40)
print(summary_df.to_string(index=False))
print("\n✅ Complete classification pipeline executed successfully!")